In [1]:
import os
import openai
api_key = os.getenv("4061_API_KEY")

In [2]:
client = openai.OpenAI(api_key = api_key)
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {"role":"system", "content":"You are a legal assistant, and your job is to find relevant case law and advice for writing legal briefs and motions."},
        #{"role":"user", "content":"Give me an overview of medical malpractice  cases involving endoscopic retrograde cholangiopancreatography (ERCP) in the state of Illinois. Cite ERCP-specific case law."}
        #{"role":"user", "content": "Give me an overview of Illinois state legislation and cases involving firearm possession."}
        {"role":"user", "content":"Give me an overview of federal disability legislation and cases."}
    ]
)


In [8]:
#gather the (potentially hallucinated) response
ans = response.choices[0].message.content
print(ans)

Federal disability legislation primarily centers around the Americans with Disabilities Act (ADA), the Rehabilitation Act of 1973, and the Individuals with Disabilities Education Act (IDEA). Here’s an overview of each of these key legislative acts and some relevant case law.

### 1. Americans with Disabilities Act (ADA)
Enacted in 1990, the ADA prohibits discrimination against individuals with disabilities in several areas including employment, public accommodations, transportation, state and local government, and telecommunications.

#### Key Provisions:
- **Title I:** Employment - Employers with 15 or more employees must provide equal employment opportunities to qualified individuals with disabilities.
- **Title II:** Public Services - Prohibits discrimination in public services (state and local government).
- **Title III:** Public Accommodations - Requires accessibility in public places like restaurants, hotels, theaters, etc.

#### Important Cases:
- **Olmstead v. L.C. (1999):** Es

In [17]:
#general one-shot self reflection
response2 = client.chat.completions.create(
    model = "gpt-4o",
    messages = [
        {"role":"system", "content":"You are a fact-checker. Determine whether the following case law exists or not. If not, attempt to find the actual cases that are being presented. If you cannot verify, say that you cannot verify."},
        {"role":"user", "content": ans}])    

In [18]:
print(response2.choices[0].message.content)

The case law mentioned in your overview is indeed well-known and established. Here's a confirmation of each case you listed:

### Americans with Disabilities Act (ADA)

1. **Olmstead v. L.C. (1999)**: This Supreme Court decision held that under the ADA, individuals with mental disabilities have the right to live in the community rather than institutions when deemed appropriate by professionals, provided the state has the resources to support such care.

2. **Toyota Motor Manufacturing, Kentucky, Inc. v. Williams (2002)**: This case clarified the definition of disability under the ADA, specifying that an individual must have an impairment that substantially limits one or more major life activities.

3. **U.S. v. Georgia (2006)**: The Supreme Court ruled that individuals could sue states for money damages under Title II of the ADA when the state violates rights protected by Title II.

### Rehabilitation Act of 1973

1. **School Board of Nassau County v. Arline (1987)**: This case extende

In [16]:
#find-fix-verify pipeline
#find step
num_it = 0
resolved = False
while not resolved and num_it < 10:
    find_step = client.chat.completions.create(
        model = "gpt-4o",
        messages = [
            {"role" : "system", "content" : "you will be given a list of legal cases. If you cannot 100% verify that a case exists and is described properly, add a [FIX] tag to it. Return the original message with these tags added."},
            {"role" : "system", "content" : ans}]
    )
    found = find_step.choices[0].message.content
    
    #fix step
    fix_step = client.chat.completions.create(
        model = "gpt-4o",
        messages = [
            {"role" : "system", "content" : "Identify any cases with the [FIX] tag. If the cases do not exist, remove them. Then, return the original message with the errors corrected."},
            {"role" : "system", "content" : find_step.choices[0].message.content}]
    )
    fixed = fix_step.choices[0].message.content
    
    #verify step
    verify_step = client.chat.completions.create(
        model = "gpt-4o",
        messages = [
            {"role" : "system", "content" : "You are a fact-checker. Ensure all legal cases listed are factual. If they are, return TRUE. Otherwise, return FALSE"},
            {"role" : "system", "content" : fixed}]
    )
    verified = verify_step.choices[0].message.content
    resolved = verified == "TRUE"

final_ans = fixed

TRUE
